In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re


# Global Variables

In [2]:
BASE_URL = "https://www.bbc.com"
START_URL = "https://www.bbc.com/urdu"
ARTICLE_LIMIT = 250   # Must be between 200–300
print("Configuration Loaded")
print("Start URL:", START_URL)
print("Article Limit:", ARTICLE_LIMIT)



Configuration Loaded
Start URL: https://www.bbc.com/urdu
Article Limit: 250


# BBC Urdu Article Link Collector

In [3]:
def collect_article_links(start_url):
    response = requests.get(start_url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/articles/" in href:
            if href.startswith("/"):
                links.add(BASE_URL + href)
            else:
                links.add(href)
    return list(links)
links = collect_article_links(START_URL)
print("Total article links found:", len(links))
print("Sample links:")
for l in links[:5]:
    print(l)


Total article links found: 43
Sample links:
https://www.bbc.com/urdu/articles/czx7gj5dg41o
https://www.bbc.com/urdu/articles/c2k8d4j82l9o
https://www.bbc.com/urdu/articles/cq6vjjn4gmdo
https://www.bbc.com/urdu/articles/cn0kpw65ylwo
https://www.bbc.com/urdu/articles/cvg14100m97o


# BBC Urdu Article Scraper (Urdu Only)

In [4]:
def scrape_urdu_article(url):
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else "N/A"

    date_tag = soup.find("time")
    date = date_tag.text.strip() if date_tag else "N/A"

    category_tag = soup.find("a", {"data-testid": "topic-link"})
    category = category_tag.text.strip() if category_tag else "N/A"

    paragraphs = soup.find_all("p")
    body = "\n".join([p.get_text(strip=True) for p in paragraphs])

    return title, date, category, body
sample_title, sample_date, sample_category, sample_body = scrape_urdu_article(links[0])

print("Title:", sample_title)
print("Date:", sample_date)
print("Category:", sample_category)
print("Body Preview:\n", sample_body[:300])


Title: عثمان طارق کا ’وقفہ‘، انڈین شائقین کرکٹ کی پریشانی اور ایشون کی وضاحت: ’یہ انڈیا کے خلاف ٹرمپ کارڈ ہو گا‘
Date: 11 فروری 2026
Category: N/A
Body Preview:
 ،تصویر کا ذریعہGetty Images
’ان بیٹرز کو تو سمجھ ہی نہیں آ رہی۔ اس کو نہیں مار پاؤ گے۔‘
’ان کو کیا، انڈیا کو بھی سمجھ نہیں آئے گی۔‘
’ایسے بولر کے خلاف پریکٹس کے لیے کس کو بلاؤں میں؟‘
’یہ انڈیا کے خلاف ان کا ٹرمپ کارڈ ہو گا۔‘
یہ وہ چند جملے ہیں جو ایک انڈین چینل پر پاکستان اور امریکہ کی ٹیم کے درمیان


# Scraping Loop + raw.txt + Metadata.json

In [5]:
metadata = {}
raw_text = ""
article_id = 1

article_links = collect_article_links(START_URL)

for link in article_links:
    if article_id > ARTICLE_LIMIT:
        break

    title, date, category, body = scrape_urdu_article(link)

    if body and len(body) > 200:
        idx = f"{article_id:03d}"

        metadata[idx] = {
            "article_id": idx,
            "url": link,
            "title": title,
            "date": date,
            "category": category
        }

        raw_text += f"{idx}\n{body}\n\n"
        article_id += 1

    time.sleep(1)

with open("Metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)

with open("raw.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)
print("Scraping completed")
print("Total articles scraped:", article_id - 1)
print("Files created:")
print("- Metadata.json")
print("- raw.txt")


Scraping completed
Total articles scraped: 43
Files created:
- Metadata.json
- raw.txt


# Diacritics Removal (Unicode-Correct)

In [6]:
def remove_diacritics(text):
    return re.sub(r'[\u064B-\u065F]', '', text)
example = "عِلم"
print("Before:", example)
print("After:", remove_diacritics(example))


Before: عِلم
After: علم


# Noise Removal (URLs, Emojis, English, Roman Urdu)

In [7]:
def remove_noise(text):
    text = re.sub(r'http\S+|www\S+', '', text)      # URLs
    text = re.sub(r'[A-Za-z]', '', text)            # English / Roman Urdu
    text = re.sub(r'[^\u0600-\u06FF\s۔؟!]', '', text)  # Non-Urdu symbols
    return text
noise_text = "بریکنگ نیوز Visit www.bbc.com "
print("Before:", noise_text)
print("After:", remove_noise(noise_text))


Before: بریکنگ نیوز Visit www.bbc.com 
After: بریکنگ نیوز   


# Sentence Segmentation (Urdu Punctuation)

In [8]:
def sentence_segmentation(text):
    sentences = re.split(r'[۔؟!]', text)
    return [s.strip() for s in sentences if s.strip()]
text = "ہوئے متاثر لوگ ہوئی بارش میں پاکستان۔ مزید بارش متوقع ہے؟"
sentences = sentence_segmentation(text)

print("Original Text:", text)
print("Segmented Sentences:")
for s in sentences:
    print("-", s)


Original Text: ہوئے متاثر لوگ ہوئی بارش میں پاکستان۔ مزید بارش متوقع ہے؟
Segmented Sentences:
- ہوئے متاثر لوگ ہوئی بارش میں پاکستان
- مزید بارش متوقع ہے


# Whitespace Normalization

In [9]:
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()
messy = "ہوئی   بارش   میں   پاکستان"
print("Before:", messy)
print("After:", normalize_whitespace(messy))


Before: ہوئی   بارش   میں   پاکستان
After: ہوئی بارش میں پاکستان


# Custom Urdu Tokenizer (From Scratch)

In [10]:
def urdu_tokenizer(sentence):
    sentence = re.sub(r'\d+', '<NUM>', sentence)
    return sentence.split()
sentence = "پاکستان میں 2024 میں بارش ہوئی"
tokens = urdu_tokenizer(sentence)

print("Sentence:", sentence)
print("Tokens:", tokens)


Sentence: پاکستان میں 2024 میں بارش ہوئی
Tokens: ['پاکستان', 'میں', '<NUM>', 'میں', 'بارش', 'ہوئی']


# Custom Urdu Stemmer (Rule-Based)

In [11]:
def urdu_stemmer(word):
    suffixes = ['وں', 'یں', 'ات', 'یاں', 'ی']
    for suf in suffixes:
        if word.endswith(suf):
            return word[:-len(suf)]
    return word
words = ["لڑکوں", "کتابیں", "مسائل"]
print("Stemming Results:")
for w in words:
    print(w, "→", urdu_stemmer(w))


Stemming Results:
لڑکوں → لڑک
کتابیں → کتاب
مسائل → مسائل


# Custom Urdu Lemmatizer

In [12]:
lemma_dict = {
    'لڑکیاں': 'لڑکی',
    'لڑکے': 'لڑکا',
    'عورتیں': 'عورت',
    'مردوں': 'مرد'
}

def urdu_lemmatizer(word):
    return lemma_dict.get(word, word)
lemma_words = ["لڑکیاں", "مردوں", "عورتیں", "کتاب"]
print("Lemmatization Results:")
for w in lemma_words:
    print(w, "→", urdu_lemmatizer(w))


Lemmatization Results:
لڑکیاں → لڑکی
مردوں → مرد
عورتیں → عورت
کتاب → کتاب


# cleaned.txt Generation (FINAL OUTPUT)

In [13]:
cleaned_text = ""

with open("raw.txt", encoding="utf-8") as f:
    content = f.read()

articles = re.split(r'\n\s*\n', content)

for article in articles:
    if not article.strip():
        continue

    lines = article.split('\n', 1)
    idx = lines[0]
    body = lines[1] if len(lines) > 1 else ""

    body = remove_diacritics(body)
    body = remove_noise(body)
    body = normalize_whitespace(body)

    sentences = sentence_segmentation(body)

    cleaned_text += f"{idx}\n"
    for sentence in sentences:
        tokens = urdu_tokenizer(sentence)
        tokens = [urdu_stemmer(urdu_lemmatizer(t)) for t in tokens]
        cleaned_text += " ".join(tokens) + "\n"
    cleaned_text += "\n"

with open("cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)
    print("cleaned.txt created successfully")
print("\nSample cleaned article:\n")

with open("cleaned.txt", encoding="utf-8") as f:
    print(f.read()[:500])



cleaned.txt created successfully

Sample cleaned article:

001
،تصویر کا ذریعہ ان بیٹرز کو تو سمجھ ہ نہ آ رہ
اس کو نہ مار پاؤ گے
ان کو کیا، انڈیا کو بھ سمجھ نہ آئے گ
ایسے بولر کے خلاف پریکٹس کے لیے کس کو بلاؤں م
یہ انڈیا کے خلاف ان کا ٹرمپ کارڈ ہو گا
یہ وہ چند جملے ہ جو ایک انڈین چینل پر پاکستان اور امریکہ ک ٹیم کے درمیان ہونے والے میچ ک کمٹر کے دوران ادا کیے گئے اور ان جمل م ذکر ہو رہا ہے پاکستان بولر عثمان طارق کا جن کا مخصوص ایکشن پہلے بھ بحث کا مرکز رہا ہے
تاہم گزشتہ روز کھیلے جانے والے میچ کے دوران اور بعد م عثمان طارق کا ایکشن اور ان کا وقفہ خصوص 


# Part **2**

# Load Cleaned Dataset

In [14]:
with open("cleaned.txt", encoding="utf-8") as f:
    corpus = f.read()

print("Cleaned corpus loaded")
print("Total characters:", len(corpus))


Cleaned corpus loaded
Total characters: 263182


# Tokenization at Corpus Level

In [15]:
tokens = corpus.split()
print("Total tokens:", len(tokens))
print("Sample tokens:", tokens[:20])


Total tokens: 62051
Sample tokens: ['001', '،تصویر', 'کا', 'ذریعہ', 'ان', 'بیٹرز', 'کو', 'تو', 'سمجھ', 'ہ', 'نہ', 'آ', 'رہ', 'اس', 'کو', 'نہ', 'مار', 'پاؤ', 'گے', 'ان']


# Unigram Model

In [16]:
from collections import defaultdict

unigram_counts = defaultdict(int)
total_tokens = 0

for token in tokens:
    unigram_counts[token] += 1
    total_tokens += 1

print("Unique unigrams:", len(unigram_counts))


Unique unigrams: 5510


# Bigram Model

In [17]:
bigram_counts = defaultdict(lambda: defaultdict(int))

for i in range(len(tokens) - 1):
    bigram_counts[tokens[i]][tokens[i+1]] += 1

print("Bigram model built")


Bigram model built


# Trigram Model

In [18]:
trigram_counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

for i in range(len(tokens) - 2):
    trigram_counts[tokens[i]][tokens[i+1]][tokens[i+2]] += 1

print("Trigram model built")


Trigram model built


# Add-k Smoothing Functions

In [19]:
VOCAB_SIZE = len(unigram_counts)
K = 1  # Add-One smoothing

def unigram_prob(word):
    return (unigram_counts[word] + K) / (total_tokens + K * VOCAB_SIZE)

def bigram_prob(w1, w2):
    return (bigram_counts[w1][w2] + K) / (sum(bigram_counts[w1].values()) + K * VOCAB_SIZE)

def trigram_prob(w1, w2, w3):
    return (trigram_counts[w1][w2][w3] + K) / (sum(trigram_counts[w1][w2].values()) + K * VOCAB_SIZE)


# Backoff Probability (Trigram → Bigram → Unigram)

In [20]:
def backoff_prob(w1, w2, w3):
    if trigram_counts[w1][w2]:
        return trigram_prob(w1, w2, w3)
    elif bigram_counts[w2]:
        return bigram_prob(w2, w3)
    else:
        return unigram_prob(w3)


# Seed Prompt Validator

In [21]:
def validate_seed(seed):
    words = seed.split()
    if len(words) < 5 or len(words) > 8:
        raise ValueError("Seed must contain 5–8 Urdu words only")
    return words

print("Seed validation ready")


Seed validation ready


# Next-Word Sampling

In [22]:
import random

def sample_next_word_bigram(prev_word):
    candidates = bigram_counts[prev_word]
    if not candidates:
        return random.choice(list(unigram_counts.keys()))
    return random.choices(
        list(candidates.keys()),
        weights=candidates.values()
    )[0]


# Trigram Sampling with Backoff

In [23]:
def sample_next_word_trigram(w1, w2):
    candidates = trigram_counts[w1][w2]
    if candidates:
        return random.choices(
            list(candidates.keys()),
            weights=candidates.values()
        )[0]
    return sample_next_word_bigram(w2)


# Article Generator (Bigram / Trigram)

In [24]:
def generate_article(seed, model="bigram", max_words=300):
    seed_words = validate_seed(seed)
    generated = seed_words[:]

    while len(generated) < max_words:
        if model == "bigram":
            next_word = sample_next_word_bigram(generated[-1])
        else:
            next_word = sample_next_word_trigram(generated[-2], generated[-1])

        generated.append(next_word)

        if generated.count('۔') >= 5 and len(generated) >= 200:
            break

    return " ".join(generated)


# Generate Bigram Articles (3)

In [25]:
seed = "پاکستان میں مہنگائی کی شرح میں اضافہ"

for i in range(3):
    article = generate_article(seed, model="bigram")
    print(f"\n--- BIGRAM ARTICLE {i+1} ---\n")
    print(article[:1200], "\n")



--- BIGRAM ARTICLE 1 ---

پاکستان میں مہنگائی کی شرح میں اضافہ دیکھنے م ڈالنے کا آغاز کیا تھا کہ دون گروپس کے ساتھ ایک سال دسی ہزار مداح کو سہارا لیتے ہ نہیں، بلکہ چیز یہ اے ک خریدار م دہل کے زمرے م قیام پذیر تھ کہ کھلاڑی ک وجہ سے حالیہ احتجاج زیادہ شدید ردعمل عموم وجہ سے جار کرنے سے لے کر د گئ ہے پاکستان نژاد کینیڈین شہر ک نظر آرہ ہے امریکہ بنگہ دیش سے بن گئ بجل ک ب ب ب ب ک آواز م کیا جائے تو سمجھ نہ کہ ہم کہ کوئ جتنا ممکن ہے ریٹینل وین شالک ویک فیلڈسکر کہت ہ انھ نے مزید کہتے ہ اور ان کے ہیڈ کوچ، سابق وزیر اعل کے ساتھ ٹوک سے اشارہ کیا گیا ہے ایک بار ہم اس علاقے یا آکسیجن ک اہلیہ اور ابراہیم کے معیا رپر پورا سامان کیمرے اور انڈیا ک تاریخ م گئے تھے لیکن موجودہ صورت ہے تاکہ اس م ہمار پالیس گرڈ سے منسلک ڈرونز ک اور پھر انھ نے تھائ گرلز لوو سیریز کا تعلق رکھنے کے ایک کرکٹ بورڈ کے تجربے پر کاروبار فنکار کو بھ کئ بار صورتحال کہ ایک کامیاب کا اصل طاقت ہے رپورٹ م پابند ہون چاہیے ب س بیرون ویب سائٹس کے نتیجے م لکھا ہے کہ زیادہ سے جڑے سوال ہے کہ آئ درج یہ احتجاج کے تحت دائر کرتے ہ انگلینڈ کے قر

# Generate Trigram Articles with Backoff (3)

In [26]:
for i in range(3):
    article = generate_article(seed, model="trigram")
    print(f"\n--- TRIGRAM ARTICLE {i+1} ---\n")
    print(article[:1200], "\n")



--- TRIGRAM ARTICLE 1 ---

پاکستان میں مہنگائی کی شرح میں اضافہ ہو رہا ہے، مگر جب پیمائش ل جات ہے پیٹرک کو محسوس ہوتا تھا جیسے کس نے منھ پر ہاتھ رکھا ہوا ہے سنہ م اس سے زیادہ آپریشن برازیل م ریکارڈ کیے گئے، جہاں ایک لاکھ روپے تک ہو سکت تھ رپورٹ کے مطابق زیادہ امکان یہ ہے کہ فنکار کیا کر سکتا ہ اور غسل خانہ صاف کرنے سے روک دیا تھا اور اس م شامل ہ جو چند انڈین صارفین نے کہ کیا گیند پھینکنے سے قبل ہ بین الاقوام تعلق ک ماہر ڈاکٹر ک ٹیم کا حصہ بنایا ہے ہمارے نامہ نگار شہزاد ملک سے ب کرتے ہوئے وضاحت د ہے جو ڈرونز یا ریڈیو ٹرانسمیشن استعمال کرنے ک سرجر کروائ تو ایک یوکرین پائلٹ نے قوم ٹیل وژن پر اس کا علاج جیل کے اندر ہ کیا جائے گا ان کا علاج جیل م قید راجپال یادو کو خود کار ہتھیار کا استعمال نظر اتا ہے کہ اس سے بھ پتہ چلتا ہے ان پر کس انسان کو دنیا م ایک اور رہائش کا کہنا ہے کہ بہتر ہو گا امیت شرما نام صارف نے لکھا ہے کہ کیا ایک چیف ایگزیکٹو ہڑتال م شرکت نہ کرنا اور یہ معاملہ کچھ ی بیان کرتے ہ جبڑے م تیز اور براہ راست سفر ک ضرورت نہ انڈین خاتون جاسوس ک کہان بھ بنایا جا رہا ہے اسرائیل کے جدی

# Generate 5 Urdu Headlines

In [27]:
def generate_headline():
    word = random.choice(list(unigram_counts.keys()))
    headline = [word]
    for _ in range(6):
        headline.append(sample_next_word_bigram(headline[-1]))
    return " ".join(headline)

print("Generated Headlines:\n")
for i in range(5):
    print("-", generate_headline())


Generated Headlines:

- جھوٹ بولے انھ دوسر جانب سے کچھ
- عکاس کرت ہ کہ اخر نتائج کا
- بجٹ کتنا ہے پروفیسر پنت نئ زندگ
- سچ نہ ہ کہ اس وقت پر
- افضل بتاتے ہ کہ کبھ اف ارٹس


# Perplexity Calculation

In [28]:
import math

def perplexity(tokens, model="bigram"):
    log_prob = 0
    N = len(tokens)

    for i in range(2, N):
        if model == "bigram":
            p = bigram_prob(tokens[i-1], tokens[i])
        else:
            p = backoff_prob(tokens[i-2], tokens[i-1], tokens[i])
        log_prob += math.log(p)

    return math.exp(-log_prob / N)

sample_tokens = tokens[:500]
print("Bigram Perplexity:", perplexity(sample_tokens, "bigram"))
print("Trigram Perplexity:", perplexity(sample_tokens, "trigram"))


Bigram Perplexity: 933.2716617527634
Trigram Perplexity: 2074.798585939672


In [29]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
